# Session 09: pandas III: group, join and reshape

Revenue by branch, category and month, and the hunt for the join that doubles revenue. Every task ends with an assert on a real number.

**SOLUTION VERSION. Tutor only.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Revenue by branch
Load `orders.csv`. Add up `order_value_ugx` for each branch and store it in `rev_branch`.

In [ ]:
import pandas as pd

orders = pd.read_csv('orders.csv')
rev_branch = orders.groupby('branch_id').order_value_ugx.sum()
print(rev_branch)

In [ ]:
assert rev_branch['B01'] == 283331305, 'Group by branch_id, then sum order_value_ugx'
assert rev_branch.sum() == 1435614185, 'The eight branches should add up to the true total'
print('Task 1 correct')

## Task 2. Branch names
Merge the branch names from `branches.csv` onto the orders with a left join. Call the result `named`.

In [ ]:
branches = pd.read_csv('branches.csv')
named = orders.merge(branches[['branch_id', 'branch_name']], on='branch_id', how='left')
print(len(orders), len(named))

In [ ]:
assert len(named) == 30229, 'A left join on one name per branch must keep every order'
assert named.branch_name.isna().sum() == 0, 'Every branch code should find a name'
print('Task 2 correct')

## Task 3. Lost rows
Join the orders to `customers.csv` on `customer_id` twice: once with `how='inner'` into `inner`, and once with `how='left'` into `left`. How many orders does the inner join lose? Store the number in `n_lost`.

In [ ]:
customers = pd.read_csv('customers.csv')
inner = orders.merge(customers, on='customer_id', how='inner')
left = orders.merge(customers, on='customer_id', how='left')
n_lost = len(orders) - len(inner)
print(n_lost)

In [ ]:
assert len(left) == 30229, 'The left join keeps every order'
assert n_lost == 9171, 'These are the walk-in orders with no customer_id'
print('Task 3 correct')

## Task 4. The join that doubles revenue
Join `orders` to `order_lines.csv` on `order_id` into `doubled`. Add up `order_value_ugx` in the joined table (`inflated`) and in `orders` (`true_total`). Print how many times too big it is.

In [ ]:
lines = pd.read_csv('order_lines.csv')
doubled = orders.merge(lines, on='order_id')
inflated = doubled.order_value_ugx.sum()
true_total = orders.order_value_ugx.sum()
print(len(doubled), inflated, true_total)
print(round(inflated / true_total, 1))

In [ ]:
assert len(doubled) == 98066, 'One row per order line is expected after the join'
assert inflated == 7606090055, 'Sum order_value_ugx in the joined table'
assert true_total == 1435614185, 'Sum order_value_ugx in orders itself'
print('Task 4 correct')

## Task 5. One table with branch, category and month
Add the product category to `lines`, then the branch and date from `orders`. Convert the date and make a `month` column such as `2024-12`. Call the table `full`.

In [ ]:
products = pd.read_csv('products.csv')
full = lines.merge(products[['product_id', 'category']], on='product_id')
full = full.merge(orders[['order_id', 'branch_id', 'order_date']], on='order_id')
full['order_date'] = pd.to_datetime(full['order_date'])
full['month'] = full.order_date.dt.strftime('%Y-%m')
print(len(full))

In [ ]:
assert len(full) == 98066, 'Both joins look up one row per key, so the row count must not change'
assert full.line_revenue_ugx.sum() == 1435614185, 'Use line_revenue_ugx, and the total must match the true revenue'
print('Task 5 correct')

## Task 6. Branch by category
Make a pivot table `pv` of `line_revenue_ugx` with branches down the side and categories across the top.

In [ ]:
pv = full.pivot_table(index='branch_id', columns='category', values='line_revenue_ugx', aggfunc='sum')
pv

In [ ]:
assert pv.shape == (8, 7), 'Expect eight branches and seven categories'
assert pv.loc['B01', 'Staples'] == 132778105, 'Check the index, columns and values'
assert pv.sum().sum() == 1435614185, 'The pivot must add up to the true total'
print('Task 6 correct')

## Task 7. Branch, category and month
Group `full` by branch, category and month together, summing `line_revenue_ugx`, into `detail`. Then store the revenue of Kampala Central (B01), Staples, December 2024 in `b01_staples_dec`.

In [ ]:
detail = full.groupby(['branch_id', 'category', 'month']).line_revenue_ugx.sum()
b01_staples_dec = detail.loc[('B01', 'Staples', '2024-12')]
print(len(detail), b01_staples_dec)

In [ ]:
assert len(detail) == 1336, 'Group by all three columns'
assert b01_staples_dec == 8099605, 'Use the tuple (B01, Staples, 2024-12)'
print('Task 7 correct')

## Task 8. Check against Sheets
Open `lakeside_for_sheets.xlsx` and build a PivotTable of revenue by branch. Its numbers come from a smaller sample, so they will differ from ours. Write one sentence below: how would you spot a doubled join in Sheets?